# Scorecard Feature Selection

Derives two feature sets from the same reproducible pipeline:

- `SCORECARD_FEATURES` - full candidate pool, keeps LendingClub's grade/rate.
- `APPLICATION_FEATURES` - excludes them. This is the set whose measured performance
  actually transfers to production, since a lender scoring its own applicants has no
  counterpart to `sub_grade` or `int_rate`.

Rerun whenever the target definition, the train window, or the WOE binning rules change.
All three changed since the last run, so the previous output is void.


In [1]:
from pathlib import Path

import pandas as pd
import polars as pl
from sklearn.linear_model import LogisticRegression

from credit_risk.data.ingestion import load_raw_accepted_loans
from credit_risk.data.target import build_target
from credit_risk.features.build_dataset import (
    assemble_feature_matrix, gbm_features, application_features,
)
from credit_risk.features.woe import WOEEncoder, rank_features_by_iv, prune_correlated_features
from credit_risk.evaluation.diagnostics import (
    coefficient_sign_report, multicollinearity_report, drop_until_signs_are_clean,
)

pl.Config.set_tbl_rows(90)

CONFIG_PATH = Path("../configs/base.yaml")
DATA_PATH = Path("../data/raw/accepted_2007_to_2018Q4.csv")


In [2]:
df = load_raw_accepted_loans(DATA_PATH)
labeled = build_target(df, CONFIG_PATH)
final = assemble_feature_matrix(labeled, CONFIG_PATH)
train = final.filter(pl.col("split") == "train")

print(train.shape, "default_rate", round(train["default_flag"].mean(), 4))


(370443, 163) default_rate 0.0892


## 1. Candidate pools


In [3]:
full_pool = gbm_features(final)
app_pool = application_features(final)
print(f"full: {len(full_pool)}  application-only: {len(app_pool)}")
print("held out of app pool:", sorted(set(full_pool) - set(app_pool)))


full: 74  application-only: 70
held out of app pool: ['grade', 'installment', 'int_rate', 'sub_grade']


## 2. IV ranking

`n_bins` is now the number of bins BEFORE monotonic merging (default 20), not the
final count. Watch the `strength` column: anything above 0.5 is a leakage alarm, not
a good feature. `sub_grade` sitting near the top is expected and is exactly why the
application-only pool exists.


In [4]:
iv_full = rank_features_by_iv(train, full_pool)
print(iv_full.to_pandas().to_string(index=False))

iv_full.write_csv("../docs/iv_ranking_full.csv")


                           feature           iv   strength
                         sub_grade 3.971021e-01     strong
                             grade 3.680637e-01     strong
                          int_rate 3.676471e-01     strong
                    fico_range_low 9.890647e-02       weak
              acc_open_past_24mths 7.729581e-02       weak
                num_tl_op_past_12m 6.595362e-02       weak
                        annual_inc 5.878589e-02       weak
                               dti 5.428426e-02       weak
                    bc_open_to_buy 5.404384e-02       weak
                   tot_hi_cred_lim 4.816926e-02       weak
                    mo_sin_rcnt_tl 4.791732e-02       weak
             mths_since_recent_inq 4.707006e-02       weak
                       avg_cur_bal 4.523179e-02       weak
                    total_bc_limit 4.187044e-02       weak
                    inq_last_6mths 4.053212e-02       weak
                       tot_cur_bal 3.889605e-02       we

In [5]:
above_threshold = iv_full.filter(pl.col("iv") >= 0.02)["feature"].to_list()
print(f"{len(above_threshold)} features with IV >= 0.02")


28 features with IV >= 0.02


## 3. Binning audit

The artefact that makes binning decisions reviewable rather than implicit.

- `n_bins == 1`: the feature carries no usable signal after merging - drop it.
- `is_monotone` False on a NUMERIC feature: should not happen; investigate if it does.
  False on a categorical feature is expected, since categories have no ordering.
- `max_abs_woe` far above ~1.5: a thin bin is dominating; check `min_bin_bads`.


In [6]:
encoder = WOEEncoder(features=above_threshold).fit(train)
report = encoder.binning_report()
print(report.to_pandas().to_string(index=False))

report.write_csv("../docs/binning_report.csv")
degenerate = report.filter(pl.col("n_bins") <= 1)["feature"].to_list()
print("single-bin features to drop:", degenerate)


              feature  is_numeric  n_bins  has_missing_bin  n_neutralised  min_bin_n  min_bin_bads is_monotone  max_abs_woe
 acc_open_past_24mths        True       8            False              0      24167          2661        True     0.465393
           annual_inc        True      11            False              0      19121          1524        True     0.459435
          avg_cur_bal        True      11             True              1         12             4        True     0.457972
       bc_open_to_buy        True      14             True              0       3451           365        True     0.651700
              bc_util        True      12             True              0       3677           392        True     0.300162
credit_history_months        True      13            False              0      18548          1422        True     0.331731
                  dti        True      11            False              0      18537          1182        True     0.381626
       f

## 4. Preliminary fit and diagnostics

This model is thrown away - it exists only to expose sign and collinearity problems.
All coefficients should be negative: WOE = ln(good/bad), so higher WOE means safer,
and a model predicting P(default=1) must weight it negatively.


In [7]:
candidates = [f for f in above_threshold if f not in degenerate]
encoder = WOEEncoder(features=candidates).fit(train)
train_woe = encoder.transform(train)
woe_cols = [f"{f}_woe" for f in candidates]

prelim_model = LogisticRegression(max_iter=1000).fit(
    train_woe.select(woe_cols).to_pandas(), train_woe["default_flag"].to_pandas()
)

print(coefficient_sign_report(prelim_model, candidates).to_string(index=False))

collinearity = multicollinearity_report(train_woe, candidates, threshold=0.6)
print(collinearity.to_string(index=False) if len(collinearity) else "no pair above 0.6")


              feature  coefficient                              flag
            sub_grade      -0.7818                                  
           annual_inc      -0.6673                                  
 acc_open_past_24mths      -0.5608                                  
  verification_status      -0.4811                                  
       home_ownership      -0.4732                                  
 mths_since_recent_bc      -0.4460                                  
                  dti      -0.3851                                  
 mo_sin_old_rev_tl_op      -0.3471                                  
mths_since_recent_inq      -0.3232                                  
      tot_hi_cred_lim      -0.2475                                  
              purpose      -0.2210                                  
     percent_bc_gt_75      -0.1715                                  
       mo_sin_rcnt_tl      -0.1617                                  
              bc_util      -0.1232

## 5. Prune correlated, then refine on signs


In [8]:
def select_features(pool: list[str], train_df: pl.DataFrame) -> list[str]:
    """IV screen -> pairwise correlation pruning -> iterative sign-based refinement."""
    ranked = rank_features_by_iv(train_df, pool)
    kept = ranked.filter(pl.col("iv") >= 0.02)["feature"].to_list()

    enc = WOEEncoder(features=kept).fit(train_df)
    kept = [f for f in kept if enc.binning_report().filter(pl.col("feature") == f)["n_bins"][0] > 1]

    woe = WOEEncoder(features=kept).fit(train_df).transform(train_df)
    corr = woe.select([f"{f}_woe" for f in kept]).to_pandas()
    corr.columns = [c.replace("_woe", "") for c in corr.columns]
    pruned = prune_correlated_features(kept, corr.corr(), threshold=0.6)

    clean, _, model = drop_until_signs_are_clean(pruned, train_df)
    print(f"  {len(pool)} pool -> {len(kept)} IV -> {len(pruned)} pruned -> {len(clean)} final")
    print(coefficient_sign_report(model, clean).to_string(index=False))
    return clean

print("SCORECARD_FEATURES (full pool)")
scorecard_features = select_features(full_pool, train)
print(scorecard_features)


SCORECARD_FEATURES (full pool)
dropping 'bc_open_to_buy' (coefficient 0.0363, still positive)
dropping 'term_months' (coefficient 0.0115, still positive)
  74 pool -> 28 IV -> 16 pruned -> 14 final
              feature  coefficient flag
            sub_grade      -0.7921     
 acc_open_past_24mths      -0.5485     
           annual_inc      -0.5381     
                  dti      -0.4644     
  verification_status      -0.4507     
       home_ownership      -0.4187     
 mo_sin_old_rev_tl_op      -0.3858     
mths_since_recent_inq      -0.3719     
 mths_since_recent_bc      -0.3064     
      tot_hi_cred_lim      -0.2549     
     percent_bc_gt_75      -0.2133     
              purpose      -0.1778     
       fico_range_low      -0.0893     
       mo_sin_rcnt_tl      -0.0401     
['sub_grade', 'fico_range_low', 'acc_open_past_24mths', 'annual_inc', 'dti', 'tot_hi_cred_lim', 'mo_sin_rcnt_tl', 'mths_since_recent_inq', 'mo_sin_old_rev_tl_op', 'mths_since_recent_bc', 'home_ownership

## 6. Application-only feature set

Same pipeline, `grade`/`sub_grade`/`int_rate`/`installment` removed. Expect a lower IV
ceiling and more features surviving the correlation prune, since `sub_grade` was
absorbing signal that other features also carry.


In [9]:
print("APPLICATION_FEATURES (no lender-derived columns)")
application_selected = select_features(app_pool, train)
print(application_selected)


APPLICATION_FEATURES (no lender-derived columns)
  70 pool -> 25 IV -> 15 pruned -> 15 final
              feature  coefficient flag
          term_months      -1.0492     
              purpose      -0.8600     
  verification_status      -0.7231     
mths_since_recent_inq      -0.6972     
           annual_inc      -0.6454     
 acc_open_past_24mths      -0.6236     
                  dti      -0.5487     
 mo_sin_old_rev_tl_op      -0.5075     
       fico_range_low      -0.4796     
       home_ownership      -0.4704     
     percent_bc_gt_75      -0.4629     
 mths_since_recent_bc      -0.4609     
      tot_hi_cred_lim      -0.4022     
       bc_open_to_buy      -0.3678     
       mo_sin_rcnt_tl      -0.2173     
['fico_range_low', 'acc_open_past_24mths', 'annual_inc', 'dti', 'bc_open_to_buy', 'tot_hi_cred_lim', 'mo_sin_rcnt_tl', 'mths_since_recent_inq', 'term_months', 'mo_sin_old_rev_tl_op', 'mths_since_recent_bc', 'home_ownership', 'purpose', 'percent_bc_gt_75', 'verificati

## 7. Output

Paste both lists into `features/build_dataset.py`, replacing the existing
`SCORECARD_FEATURES` and adding `APPLICATION_FEATURES`. Production code consumes the
decision; it must never re-derive it at runtime.


In [10]:
print("SCORECARD_FEATURES = [")
print("    " + ", ".join(f'"{f}"' for f in scorecard_features))
print("]")
print()
print("APPLICATION_FEATURES = [")
print("    " + ", ".join(f'"{f}"' for f in application_selected))
print("]")


SCORECARD_FEATURES = [
    "sub_grade", "fico_range_low", "acc_open_past_24mths", "annual_inc", "dti", "tot_hi_cred_lim", "mo_sin_rcnt_tl", "mths_since_recent_inq", "mo_sin_old_rev_tl_op", "mths_since_recent_bc", "home_ownership", "purpose", "percent_bc_gt_75", "verification_status"
]

APPLICATION_FEATURES = [
    "fico_range_low", "acc_open_past_24mths", "annual_inc", "dti", "bc_open_to_buy", "tot_hi_cred_lim", "mo_sin_rcnt_tl", "mths_since_recent_inq", "term_months", "mo_sin_old_rev_tl_op", "mths_since_recent_bc", "home_ownership", "purpose", "percent_bc_gt_75", "verification_status"
]
